# 🔬 Certificados de Obstrução em 32 Variáveis: HRSBF Cúbicas
## Verificação Independente de Fibras Afins e Quocientes de Órbitas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/contatofilipevono-rgb/hrsbf-bent-verification/blob/main/avanco_t32/Certificados_T32_Colab.ipynb)

Este notebook executa a **auditoria computacional independente e reproduzível** dos certificados de não-bentness obtidos para famílias de funções Booleanas cúbicas homogêneas simétricas por rotação em $n=32$ variáveis.

### Roteiro de Verificação:
1. **Certificado Curto de 7 Fibras ($h = 0x1$)**: Demonstra que 7 condições necessárias somam $0 = 1$ em $\mathbb{F}_2$, excluindo incondicionalmente $2^{120}$ funções sem enumeração individual (~2,3 s).
2. **Obstrução da Fibra Antidiagonal ($J = (1,\dots,1)$)**: Prova a existência de 6 formas lineares independentes, excluindo um subespaço de dimensão 149 ($2^{149}$ funções) (~0,5 s).
3. **Auditoria Ampliada das 631 Classes de Quociente (Peso $\le 2$)**: Confirmação direta sobre as ANFs originais de 30.038 equações, excluindo 607 classes (~13 s).
4. **Análise Estrutural das 24 Classes Restantes**: Identificação do subespaço de dimensão 7 de triplas de mesma paridade e caracterização da barreira de posto polar $\le 12$.

> **Nota de Escopo & Sobriedade Científica:** Trata-se de certificados parciais assistidos por computação para famílias específicas em 32 variáveis. **Não resolve o caso geral $n=32$**, não possui formalização Lean e a originalidade perante a literatura recente (ex.: Sun et al., 2026) permanece em avaliação.

In [ ]:
# Setup de ambiente: compatível com execução local e Google Colab
import sys, os
from pathlib import Path

if 'google.colab' in sys.modules:
    print('[COLAB] Detectado ambiente Google Colab!')
    if not Path('verificar_7_fibras.py').exists():
        print('[INFO] Clonando repositório para obter scripts e dados...')
        os.system('git clone --depth 1 https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git')
        if Path('hrsbf-bent-verification/avanco_t32').exists():
            os.chdir('hrsbf-bent-verification/avanco_t32')
    else:
        print('[OK] Arquivos de dados já presentes no diretório de trabalho.')
else:
    print('[LOCAL] Ambiente local detectado.')

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print('Python:', sys.version.split()[0])
print('Diretório:', Path.cwd())


---
### Etapa 1: Certificado de 7 Fibras ($h = 0x1$, Exclusão de $2^{120}$ Funções)

Reavalia as matrizes polares e linhas do certificado diretamente a partir das ANFs originais de 32 variáveis (sem importar rotinas de busca).
Demonstra que a soma em $\mathbb{F}_2$ das 7 condições de balanceamento satisfaz:
$$\sum_{i=1}^7 L_i(c) = 0 \quad (\forall c \in \mathbb{F}_2^{155}), \quad \text{mas bentness exige } L_i(c) = 1 \implies 0 = 1.$$

In [ ]:
import verificar_7_fibras
verificar_7_fibras.run()


---
### Etapa 2: Obstrução da Fibra Antidiagonal ($J = (1, 1, \dots, 1)$)

Demonstra que a rotação sob $J$ força $q_J$ a ser afim-constante se $B_J = 0$. Obtém 6 formas lineares independentes, definindo um núcleo de dimensão 149 ($2^{149}$ funções excluídas).

In [ ]:
import runpy
runpy.run_path('verificar_fibra_antidiagonal.py', run_name='__main__')


---
### Etapa 3: Auditoria Estendida das 631 Classes de Quociente (Peso $\le 2$)

Examina todas as $1 + 35 + \binom{35}{2} = 631$ classes de quociente no espaço de 35 parâmetros quadráticos.
Verifica diretamente 30.038 equações sobre as ANFs originais.
- **19 classes excluídas** por $B_J = 0$ (fibra antidiagonal).
- **588 classes excluídas** por contradições lineares comprovadas.
- **Total excluído:** 607 classes de 631 (cada classe representa $2^{120}$ funções originais).

In [ ]:
import verificar_classes
verificar_classes.main()


---
### Etapa 4: Análise das 24 Classes Restantes (Subespaço de Mesma Paridade)

As 24 classes não excluídas pelo teste de posto 14 residem em um subespaço de dimensão 7 de triplas de mesma paridade. Nesse subespaço, as formas polares se separam nos blocos par e ímpar de dimensão 8, limitando o posto a no máximo 12.

In [ ]:
import json
from pathlib import Path

res_rem = json.loads(Path('estrutura_remanescente.json').read_text(encoding='utf-8'))
print('Estrutura das 24 classes remanescentes:')
print(f"- Representantes no subespaço de mesma paridade ({len(res_rem['parity_orbit_representatives'])} geradores em n=16):")
for rep in res_rem['parity_orbit_representatives']:
    print(f"  {rep} -> índices pares decimados {tuple(i//2 for i in rep)} em n=8")
print(f"- Classes não excluídas por posto 14: {res_rem['remaining_sparse_quotients']}")
print(f"- Posto da contração J no subespaço: {res_rem['ones_contraction_rank_on_subspace']}")
print(f"- Excluídas pela fibra J em 128: {res_rem['ones_fiber_excludes_of_128']}")


---
### 📋 Resumo Consolidado e Hashes Criptográficos SHA-256

In [ ]:
import hashlib, json
from pathlib import Path
print('Hashes SHA-256 dos artefatos de dados e certificados:')
for fname in ['certificado_7_fibras.json', 'fibra_J_certificado.json', 'resultado_631_classes.json', 'estrutura_remanescente.json']:
    p = Path(fname)
    if p.exists():
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        print(f'{fname:32} SHA-256: {h}')
print('\n[CONCLUIDO] Auditoria computacional de t=32 concluida com sucesso e 100% reproduzivel!')
